In [ ]:
import requests
r = requests.get("https://grobidorg-grobid.hf.space/api/isalive", timeout=30)
print("Status:", r.status_code)
print("Content-Type:", r.headers.get("content-type"))
print("Body:", r.text[:200])

Status: 200
Content-Type: text/plain
Body: true


In [10]:
import os

# Clear previous run so all 6 get reprocessed
for f in os.listdir("PDF_extracted_txt_Grobid"):
    os.remove(os.path.join("PDF_extracted_txt_Grobid", f))
if os.path.exists("grobid_text_extraction_log.csv"):
    os.remove("grobid_text_extraction_log.csv")

In [ ]:
GROBID_URL = "https://grobidorg-grobid.hf.space/api/processFulltextDocument"

In [14]:
"""
grobid_text_extraction.py

Sends each PDF to the GROBID public API (no Docker or local
installation needed) and saves ONE .txt file per paper into
PDF_extracted_txt_Grobid/.

Public API endpoint used:
    "https://grobidorg-grobid.hf.space/api/processFulltextDocument"

This is the official GROBID demo instance hosted on HuggingFace
Spaces. It has a rate limit (roughly 1 request per second is safe),
so this script adds a small delay between requests automatically.
For a corpus of 81 papers this adds about 2 minutes total, for 12,000
papers you would want to run a local Docker instance instead.

Python dependencies:
    pip install requests lxml
"""

import csv
import os
import re
import time

import requests
from lxml import etree

PDF_FOLDER = "Test"
OUTPUT_FOLDER = "PDF_extracted_txt_Grobid"
LOG_PATH = "grobid_text_extraction_log.csv"
FLAGGED_LOG_PATH = "grobid_text_extraction_flagged.csv"

GROBID_URL = "https://grobidorg-grobid.hf.space/api/processFulltextDocument"
REQUEST_DELAY_SECONDS = 1.5  # stay comfortably under the public rate limit
MIN_CHARS_LIKELY_OK = 3000

TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}
TEI_P = "{http://www.tei-c.org/ns/1.0}p"

# Map GROBID's own <head> section-title text onto the same taxonomy
# used across the rest of this pipeline. Matched against GROBID's
# already-correct div boundaries, not raw, ambiguous lines.
SECTION_PATTERNS = {
    "intro":              r"introduction",
    "methods":            r"material[s]?\s+and\s+method[s]?|method[s]?",
    "results_discussion": r"result[s]?\s+and\s+discussion|result[s]?\s*&\s*discussion",
    "results":            r"result[s]?",
    "discussion":         r"discussion",
    "conclusion":         r"conclusion[s]?",
}
HEADER_NUMBER_PREFIX = re.compile(r"^\d+(\.\d+)*[\.\)]?\s*")


def classify_div_head(head_text: str) -> str:
    if not head_text:
        return "other"
    candidate = HEADER_NUMBER_PREFIX.sub("", head_text.strip()).strip()
    for name, pattern in SECTION_PATTERNS.items():
        if re.fullmatch(pattern, candidate, flags=re.IGNORECASE):
            return name
    return "other"


def doi_to_safe_id(doi: str) -> str:
    return doi.strip().replace(".", "-").replace("/", "_")


def call_grobid(pdf_path: str) -> str:
    """Send one PDF to the GROBID public API, return raw TEI XML."""
    with open(pdf_path, "rb") as f:
        response = requests.post(
            GROBID_URL,
            files={"input": f},
            data={"consolidateHeader": "1", "consolidateCitations": "0"},
            timeout=180,
        )
    response.raise_for_status()
    return response.text


def parse_tei(tei_xml: str) -> dict:
    """Extract DOI, abstract, and body sections from GROBID's TEI output.

    Deliberately never touches <back>, where GROBID puts the
    bibliography, so reference content structurally cannot leak into
    the output here.

    Many papers use subsection-style headers like 'Preparation of
    defatted groundnut flour' or 'Protein solubility' rather than a
    single top-level 'Materials and Methods' or 'Results' heading.
    GROBID correctly detects these as separate divs but their head
    text doesn't match the SECTION_PATTERNS, so they get classified
    as 'other'. A position-aware fallback resolves this: after
    collecting all divs in document order, any 'other' div that
    appears before the first results-type section is reassigned to
    'methods', and any 'other' div that appears after is reassigned
    to 'results_discussion'. This correctly handles most food science
    papers where Methods and Results are broken into many named
    subsections rather than one top-level header.
    """
    root = etree.fromstring(tei_xml.encode("utf-8"))

    doi_el = root.find(".//tei:idno[@type='DOI']", TEI_NS)
    doi = doi_el.text.strip() if doi_el is not None and doi_el.text else None

    sections = {}

    abstract_el = root.find(".//tei:abstract", TEI_NS)
    if abstract_el is not None:
        abstract_text = " ".join(p.text or "" for p in abstract_el.iter(TEI_P))
        if abstract_text.strip():
            sections["abstract"] = abstract_text.strip()

    body = root.find(".//tei:text/tei:body", TEI_NS)
    if body is None:
        return {"doi": doi, "sections": sections}

    RESULTS_TYPES = {"results", "results_discussion", "discussion"}

    # First pass: collect divs in document order with their classified name
    div_list = []
    for div in body.findall("tei:div", TEI_NS):
        head_el = div.find("tei:head", TEI_NS)
        head_text = head_el.text if head_el is not None else None
        section_name = classify_div_head(head_text)
        paragraphs = [p.text or "" for p in div.iter(TEI_P)]
        div_text = "\n\n".join(p.strip() for p in paragraphs if p.strip())
        if div_text:
            div_list.append((section_name, div_text, head_text or ""))

    # Second pass: find where results-type content first appears
    first_results_idx = next(
        (i for i, (name, _, _) in enumerate(div_list) if name in RESULTS_TYPES),
        len(div_list),  # if no results section found, treat everything as methods
    )

    # Third pass: reassign 'other' based on position, then accumulate
    for i, (section_name, div_text, head_text) in enumerate(div_list):
        if section_name == "other":
            section_name = "methods" if i < first_results_idx else "results_discussion"
        sections[section_name] = (sections.get(section_name, "") + "\n\n" + div_text).strip()

    return {"doi": doi, "sections": sections}


def write_sections_to_txt(doi: str, sections: dict, output_path: str):
    """Write the parsed sections to a plain, human-readable .txt file
    with unambiguous ===== markers between sections."""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"DOI: {doi or '(not found)'}\n")
        for section_name, section_text in sections.items():
            f.write(f"\n===== {section_name} =====\n")
            f.write(section_text)
            f.write("\n")


# ---------------------------------------------------------------------------
# Crash resilience, same pattern as pdf_text_extraction.py
# ---------------------------------------------------------------------------
def load_processed_log(log_path: str) -> set:
    done = set()
    if not os.path.exists(log_path):
        return done
    with open(log_path, "r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            done.add(row["source_file"])
    return done


def append_log(log_path: str, source_file: str, output_file: str, doi: str):
    is_new = not os.path.exists(log_path)
    with open(log_path, "a", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["source_file", "output_file", "doi"])
        writer.writerow([source_file, output_file, doi])


def run():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    already_done = load_processed_log(LOG_PATH)
    if already_done:
        print(f"Resuming: {len(already_done)} PDFs already extracted, will be skipped.")

    pdf_files = [
        f for f in sorted(os.listdir(PDF_FOLDER))
        if f.lower().endswith(".pdf") and f not in already_done
    ]
    print(f"Found {len(pdf_files)} PDFs left to extract in {PDF_FOLDER}/")

    if not pdf_files:
        print("Nothing new to extract.")
        return

    print(f"Using GROBID public API: {GROBID_URL}")
    print(f"Rate limit delay: {REQUEST_DELAY_SECONDS}s between requests")
    print()

    failed = []
    flagged = []
    ok_count = 0

    for i, fname in enumerate(pdf_files, 1):
        pdf_path = os.path.join(PDF_FOLDER, fname)
        print(f"[{i}/{len(pdf_files)}] {fname}")
        try:
            tei_xml = call_grobid(pdf_path)
            parsed = parse_tei(tei_xml)
            doi = parsed["doi"]
            safe_id = doi_to_safe_id(doi) if doi else os.path.splitext(fname)[0]

            output_path = os.path.join(OUTPUT_FOLDER, f"{safe_id}.txt")
            write_sections_to_txt(doi, parsed["sections"], output_path)

            append_log(LOG_PATH, fname, f"{safe_id}.txt", doi or "")

            char_count = sum(len(t) for t in parsed["sections"].values())
            note = doi if doi else "(DOI not found, used PDF filename instead)"

            if char_count < MIN_CHARS_LIKELY_OK:
                flagged.append((fname, safe_id, char_count))
                print(f"    doi={note}  -> {safe_id}.txt  ({char_count} chars, FLAGGED: unusually short)")
            else:
                ok_count += 1
                print(f"    doi={note}  -> {safe_id}.txt  ({char_count} chars)")
            time.sleep(REQUEST_DELAY_SECONDS)
        except Exception as e:
            print(f"    [ERROR] {e}")
            failed.append(fname)

    print()
    print("=" * 60)
    print("GROBID extraction summary for this run:")
    print(f"  Total PDFs processed:  {len(pdf_files)}")
    print(f"  Likely extracted OK:   {ok_count}")
    print(f"  Flagged (short, needs review): {len(flagged)}")
    print(f"  Failed with an error:  {len(failed)}")
    print("=" * 60)

    if flagged:
        is_new = not os.path.exists(FLAGGED_LOG_PATH)
        with open(FLAGGED_LOG_PATH, "a", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            if is_new:
                writer.writerow(["source_file", "output_file", "char_count"])
            for fname, safe_id, char_count in flagged:
                writer.writerow([fname, f"{safe_id}.txt", char_count])
        print(f"Flagged files written to {FLAGGED_LOG_PATH}, review these specifically.")

    if failed:
        print(f"Failed files: {failed}")

    print(f"\nDone. Extracted text saved in {OUTPUT_FOLDER}/")


if __name__ == "__main__":
    run()

Found 6 PDFs left to extract in Test/
Using GROBID public API: https://grobidorg-grobid.hf.space/api/processFulltextDocument
Rate limit delay: 1.5s between requests

[1/6] 10-1007_s13197-015-1758-7.pdf
    doi=10.1007/s13197-015-1758-7  -> 10-1007_s13197-015-1758-7.txt  (13052 chars)
[2/6] 10-1080_19476337-2017-1301554.pdf
    doi=10.1080/19476337.2017.1301554  -> 10-1080_19476337-2017-1301554.txt  (15722 chars)
[3/6] 10-3389_fnut-2025-1708593.pdf
    doi=10.3389/fnut.2025.1708593  -> 10-3389_fnut-2025-1708593.txt  (23755 chars)
[4/6] 10-3390_app142411979.pdf
    doi=10.3390/app142411979  -> 10-3390_app142411979.txt  (13712 chars)
[5/6] 10-3390_foods10020281.pdf
    doi=10.3390/foods10020281  -> 10-3390_foods10020281.txt  (17752 chars)
[6/6] 10-5433_1679-0359-2025v46n3p675.pdf
    doi=10.5433/1679-0359.2025v46n3p675  -> 10-5433_1679-0359-2025v46n3p675.txt  (17722 chars)

GROBID extraction summary for this run:
  Total PDFs processed:  6
  Likely extracted OK:   6
  Flagged (short, need